# CREADS reproduction

Run the cells in order and upload `CREADS_Colab_Repro_Inputs.zip` when prompted.
The notebook includes model estimation and analyses of the supplied intermediate results.
See `docs/reproduction.md` for the execution environment and scope.

In [ ]:
#@title S0. Setup — install & folders  (Runtime > Run all to execute step by step)
!pip -q install openpyxl==3.1.5 scikit-learn==1.5.2 pandas==2.2.3 numpy==1.26.4 scipy==1.14.1 matplotlib==3.9.2 seaborn==0.13.2
import os, json, numpy as np, pandas as pd
for d in ('/content/creads', '/content/creads/data', '/content/creads/code', '/content/creads/colab', '/content/creads/outputs'):
    os.makedirs(d, exist_ok=True)
print('Next: run cell S1 and upload CREADS_Colab_Repro_Inputs.zip (ONE file) when the picker opens.')


In [ ]:
#@title S1. Inputs — upload the repro bundle (ONE zip; data + code)
import os, shutil, zipfile
BASE = '/content/creads'
NEED = ['data/MATCHED_340REGIONS_WITH_GDL_2023.xlsx',
        'data/COMPLETE_584REGIONS_WITH_CONNECTIVITY_2023.xlsx',
        'code/creads_core.py', 'code/creads_core_improved.py',
        'code/NUMBERS_FINAL.json']
def _missing():
    return [p for p in NEED if not os.path.exists(f'{BASE}/{p}')]
if _missing():
    from google.colab import files
    print('Upload ONE file: CREADS_Colab_Repro_Inputs.zip (the repro bundle).')
    up = files.upload()
    for name in up:
        base = os.path.basename(name)
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as z:   # entries already carry the creads/ prefix
                z.extractall('/content')
        elif name.endswith('.ipynb'):
            shutil.move(name, f'{BASE}/colab/{base}')
        elif name.endswith(('.py', '.json')):
            shutil.move(name, f'{BASE}/code/{base}')
        else:
            shutil.move(name, f'{BASE}/data/{base}')
    # repair pass: anything flat in BASE (e.g. uploaded as loose files) is routed by extension
    for name in list(os.listdir(BASE)):
        p = f'{BASE}/{name}'
        if os.path.isfile(p):
            if name.endswith('.ipynb'): shutil.move(p, f'{BASE}/colab/{name}')
            elif name.endswith(('.py', '.json')): shutil.move(p, f'{BASE}/code/{name}')
            elif name.endswith('.xlsx'): shutil.move(p, f'{BASE}/data/{name}')
miss = _missing()
if miss:
    raise FileNotFoundError(
        'Bundle files still missing after upload: %s\n'
        'Re-run this S1 cell and upload CREADS_Colab_Repro_Inputs.zip (the whole bundle, one file).' % miss)
print('All required inputs present.')
!ls -la /content/creads/data /content/creads/code


In [ ]:
#@title S2. Step 1 — feature engineering (single pinned protocol; seed 2024/42)
import os, sys
if not os.path.exists('/content/creads/code/creads_core.py'):
    raise FileNotFoundError('creads_core.py not found in /content/creads/code - '
                            'run cell S1 first and upload CREADS_Colab_Repro_Inputs.zip when prompted.')
sys.path.insert(0, '/content/creads/code')
import creads_core as cc, creads_core_improved as ci
cc.DATA_DIR = '/content/creads/data'
cc.FILE_98 = f'{cc.DATA_DIR}/MATCHED_340REGIONS_WITH_GDL_2023.xlsx'
cc.FILE_300 = f'{cc.DATA_DIR}/COMPLETE_584REGIONS_WITH_CONNECTIVITY_2023.xlsx'
df98, df300 = cc.step1_load(verbose=False)
df98, df300, _ = ci.step2_features_improved(df98, df300, verbose=False)
FEATS = [f for f in ci.FEATS_IMPROVED if f in df98.columns]
# fail-fast pins: the protocol must reproduce the manuscript sample exactly
assert len(df300) == 584, 'inventory must be 584 regions'
assert int(df98.HDI.notna().sum()) == 340, 'labelled sample must be 340 regions'
assert len(FEATS) == 24, 'the deployed model uses the 24-variable matrix'
print('regions:', len(df300), '| labelled:', int(df98.HDI.notna().sum()), '| features:', len(FEATS))
df300.to_csv('/content/creads/outputs/s2_inventory_features.csv', index=False)
df98.to_csv('/content/creads/outputs/s2_matched_features.csv', index=False)


In [ ]:
#@title S3. Step 2 — single-level GMM partitions (spatial-only working partition; K sweep)
# This cell fits the notebook's working single-level partition on three accessibility
# features (median imputation, standardisation, full-covariance GMM, seed 2024, n_init=20)
# and keeps it available to downstream cells via df300['regime'] / df300['regime_label'].
# Terminology note (final version): the manuscript's deployed regimes are the two-level,
# outcome-free geospatial regimes (Step 4). A single-level partition survives only as the
# EARLIER single-level partition, on which the Table V stress-test grid is computed; the
# manuscript's legacy definition is welfare-integrated (its cluster vector contained the
# cross-fitted predicted HDI) and is reproduced in S7b once HDI_predicted exists.
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, normalized_mutual_info_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
FEATS3 = ['border_distance', 'log_pop_density', 'log_dist_capital']
X = SimpleImputer(strategy='median').fit_transform(df300[FEATS3].values.astype(float))
Z = StandardScaler().fit_transform(X)
rows = {}
for K in range(2, 7):
    gm = GaussianMixture(n_components=K, covariance_type='full', random_state=2024, n_init=20).fit(Z)
    lab = gm.predict(Z)
    rows[K] = dict(BIC=gm.bic(Z), AIC=gm.aic(Z), silhouette=silhouette_score(Z, lab))
pd.DataFrame(rows).T.round(4).to_csv('/content/creads/outputs/s3_k_sweep.csv')
gm3 = GaussianMixture(n_components=3, covariance_type='full', random_state=2024, n_init=20).fit(Z)
lab3 = gm3.predict(Z)
df300['regime'] = lab3
order = pd.Series(df300.nightlight.values).groupby(lab3).mean().sort_values().index.tolist()
remap = {o: n for o, n in zip(order, ['Remote-Peripheral', 'Semi-Accessible', 'Core-Accessible'])}
df300['regime_label'] = df300.regime.map(remap)
j = df300[df300.rid.isin(df98.loc[df98.HDI.notna(), 'rid'])][['rid', 'regime_label']].merge(
    df98.loc[df98.HDI.notna(), ['rid', 'HDI', 'viirs_mean', 'pop_density']], on='rid')
T2 = df300.groupby('regime_label').agg(N=('rid', 'size'), viirs=('nightlight', 'mean'),
        popd=('pop_density', 'mean'), pop=('pop_sum', 'sum'))
T2['share_pct'] = T2.N / T2.N.sum() * 100
T2['hdi_obs'] = j.groupby('regime_label').HDI.mean()
T2['n_labelled'] = j.groupby('regime_label').size()
T2.round(3).to_csv('/content/creads/outputs/s3_table2.csv')
audit = dict(k_sweep={k: round(v['silhouette'], 4) for k, v in rows.items()},
             k_sweep_bic={k: round(v['BIC'], 2) for k, v in rows.items()},
             table2_sizes=T2.N.to_dict(), table2_sil=round(rows[3]['silhouette'], 4),
             table2_variant='spatial-only 3-feature GMM — notebook working partition, not a manuscript table',
             nmi_country=round(normalized_mutual_info_score(df300.ISO3.astype(str), lab3), 4),
             purity=round(pd.crosstab(lab3, df300.ISO3).max(axis=1).sum() / len(lab3), 4))
json.dump(audit, open('/content/creads/outputs/s3_audit.json', 'w'), indent=1)
print(pd.DataFrame(rows).T.round(4)); print(T2.round(3))
print('NMI(country) =', audit['nmi_country'], '| purity =', audit['purity'])
print('NOTE: spatial-only working partition — the manuscript legacy diagnostics')
print('(silhouette highest at K = 2 = 0.3260; BIC monotone to K = 6; welfare-integrated')
print('K = 3) are reproduced in S7b, after HDI_predicted is built in S7.')


In [ ]:
#@title S4. Step 3 — bootstrap stability (single-level, country, regional; 200 resamples)
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score
rng = np.random.default_rng(11)
def ari_proto(Xm, K, point, B=200):
    aris = []
    for b in range(B):
        idxb = rng.integers(0, len(Xm), len(Xm))
        lb = GaussianMixture(n_components=K, covariance_type='full', random_state=2024,
                             n_init=10).fit(Xm[idxb]).predict(Xm[np.unique(idxb)])  # compare on covered rows only
        uniq = np.unique(idxb); lp = point[uniq]
        cont = np.zeros((K, K))
        for a in range(K):
            for b2 in range(K): cont[a, b2] = np.sum((lb == a) & (lp == b2))
        ra, cb = linear_sum_assignment(-cont)
        mp = {int(a): int(b2) for a, b2 in zip(ra, cb)}
        aris.append(adjusted_rand_score(lp, np.array([mp.get(int(x), -1) for x in lb])))
    return round(float(np.median(aris)), 3), round(float(np.percentile(aris, 5)), 3)
res = {}
res['single_level'] = ari_proto(Z, 3, gm3.predict(Z))
# country + regional levels via the archived extension module
import extension_ms3  # noqa  (runs the two-level extension end-to-end)
E = json.load(open('/content/creads/code/NUMBERS_EXT.json'))
res['country_level'] = (E['ari_country']['median'], E['ari_country']['p05'])
res['regional_level'] = (E['ari_regional']['median'], E['ari_regional']['p05'])
pd.DataFrame(res, index=['median_ARI', 'p05_ARI']).to_csv('/content/creads/outputs/s4_bootstrap_ari.csv')
print(pd.DataFrame(res, index=['median_ARI', 'p05_ARI']))


In [ ]:
#@title S5. Step 4 — two-level extension (Tables II-B / II-C / II-D)
SWP = pd.read_csv('/content/creads/outputs_ext/csv_exports/ext_country_sweep.csv')
MEM = pd.read_csv('/content/creads/outputs_ext/csv_exports/ext_country_membership.csv')
CS = pd.read_csv('/content/creads/outputs_ext/csv_exports/ext_cell_summary.csv')
RSW = pd.read_csv('/content/creads/outputs_ext/csv_exports/ext_regional_sweep.csv')
print('TABLE II-B (country K sweep)'); print(SWP.round(3))
print('TABLE II-C (cluster composition)'); print(MEM.groupby('cluster').ISO3.agg(['count', ','.join]))
print('TABLE II-D (regime cells)'); print(CS.round(3))
for nm, df_ in [('iib_sweep', SWP), ('iid_cells', CS)]:
    df_.to_csv(f'/content/creads/outputs/s5_{nm}.csv', index=False)
print('K* =', E['K_star'], '| n_cells =', E['n_cells'])


In [ ]:
#@title S5b. K*-sensitivity of the extension (Table II-E)
# Compare the guard-admissible partitions (K = 2 vs K* = 4) and the inadmissible
# candidates on content, sufficiency and stability criteria. Run S5 first.
import numpy as np, pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.impute import SimpleImputer
from scipy.optimize import linear_sum_assignment
# country-level matrix (identical to the extension module's construction)
gC = df300.groupby('ISO3')
C = pd.DataFrame({
    'popd': np.log1p(gC['pop_density'].mean().clip(lower=0)),
    'urban': gC['urban_ratio'].mean(),
    'border': np.log1p(gC['border_distance'].mean().clip(lower=0)),
    'cap': np.log1p(gC['dist_to_capital_km'].mean().clip(lower=0)),
    'air': np.log1p(gC['dist_to_nearest_airport_km'].mean().clip(lower=0)),
    'smod': gC['smod_urban_ratio'].mean(),
    'tree': gC['tree_cover'].mean()})
CF = ["popd", "urban", "border", "cap", "air", "smod", "tree"]
Xc_ = C[CF].values.astype(float)
Xcs = (Xc_ - Xc_.mean(0)) / (Xc_.std(0) + 1e-9)
feats3 = ['border_distance', 'log_pop_density', 'log_dist_capital']
NAMES = {2: ['Sparse-Interior', 'Dense-Core'], 3: ['Sparse-Interior', 'Transitional-Access', 'Dense-Core']}
rngK = np.random.default_rng(11)
rows = []
for K in [2, 3, 4, 5]:
    raw = GaussianMixture(n_components=K, covariance_type='full', random_state=2024, n_init=20).fit(Xcs).predict(Xcs)
    minc = int(pd.Series(raw).value_counts().min())
    order = pd.Series(C['popd'].values).groupby(raw).mean().sort_values().index.tolist()
    remap = {o: i + 1 for i, o in enumerate(order)}
    df300['cluster'] = df300.ISO3.map(dict(zip(C.index, [remap[r] for r in raw])))
    aris = []
    for b in range(100):
        idxb = rngK.integers(0, len(Xcs), len(Xcs)); uniq = np.unique(idxb)
        lb = GaussianMixture(n_components=K, covariance_type='full', random_state=2024, n_init=10).fit(Xcs[idxb]).predict(Xcs[uniq])
        lp = np.array([remap[raw[i]] for i in uniq])
        cont = np.zeros((K, K))
        for a in range(K):
            for b2 in range(K): cont[a, b2] = np.sum((lb == a) & (lp == b2))
        ra, cb = linear_sum_assignment(-cont)
        mp = {int(a): int(b2) for a, b2 in zip(ra, cb)}
        aris.append(adjusted_rand_score(lp, np.array([mp.get(int(x), -1) for x in lb])))
    labels = np.empty(len(df300), dtype=object)
    for c in range(1, K + 1):
        sub = df300[df300.cluster == c]
        Xr = SimpleImputer(strategy='median').fit_transform(sub[feats3].values.astype(float))
        Zr = (Xr - Xr.mean(0)) / (Xr.std(0) + 1e-9)
        cands = []
        for Kr in (2, 3):
            gk = GaussianMixture(n_components=Kr, covariance_type='full', random_state=2024, n_init=20).fit(Zr)
            lk = gk.predict(Zr)
            cands.append((gk.bic(Zr), Kr, lk, int(pd.Series(lk).value_counts().min())))
        feas = [t for t in cands if t[3] >= 8]
        _, Kr, lk, _ = sorted(feas or cands, key=lambda t: t[0])[0]
        m = df300.cluster == c
        ordt = pd.Series(sub.pop_density.values).groupby(lk).mean().sort_values().index.tolist()
        rm = {o: i for i, o in enumerate(ordt)}
        labels[m] = [f'C{c}-{NAMES[Kr][rm[l]]}' for l in lk]
    df300['cell'] = labels
    jj = df300[['rid', 'cell']].merge(df98.loc[df98.HDI.notna(), ['rid', 'HDI']], on='rid', how='inner')
    obs = jj.groupby('cell').HDI.mean(); nlab = jj.groupby('cell').size()
    tt = pd.DataFrame({'hdi': obs, 'type': [k.split('-', 1)[1] for k in obs.index]})
    sp = [g.hdi.max() - g.hdi.min() for t, g in tt.groupby('type') if len(g) >= 2]
    rows.append(dict(K=K, admissible=minc >= 3, cells=len(obs), lab12=int((nlab >= 12).sum()),
                     spread_max=round(max(sp), 3) if sp else None,
                     ari=round(float(np.median(aris)), 3), ari_p05=round(float(np.percentile(aris, 5)), 3)))
T2E = pd.DataFrame(rows); T2E.to_csv('/content/creads/outputs/s5b_ksensitivity.csv', index=False)
print(T2E)
print('Verdict: coarsening to K = 2 halves the within-type contrast and the FPCA-eligible')
print('cells without a stability gain; K* = 4 is retained.')


In [ ]:
#@title S6. Step 5 — matched ablation on identical folds (Table III protocol, 8 variants)
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
SEED = 2024
v = df98['HDI'].notna().values
Xm = df98.loc[v, FEATS].values.astype(float)
y = df98.loc[v, 'HDI'].values.astype(float)
g = df98.loc[v, 'ISO3'].values
reg = df98.loc[v, 'rid'].map(dict(zip(df300.rid, df300.regime_label))).fillna('Semi-Accessible').values
def fit_ens(Xtr, ytr):
    im = SimpleImputer(strategy='median').fit(Xtr); Xt = im.transform(Xtr)
    rf = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=2, n_jobs=-1,
                               random_state=SEED).fit(Xt, ytr)
    gb = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                                   subsample=0.8, random_state=SEED).fit(Xt, ytr)
    rd = Ridge(alpha=1.0).fit(StandardScaler().fit_transform(Xt), ytr)
    s = StandardScaler().fit(Xt)
    w = np.clip([rf.score(Xt, ytr), gb.score(Xt, ytr),
                 rd.score(s.transform(Xt), ytr)], 0, None); w = w / w.sum()
    return im, s, rf, gb, rd, w
def pred_ens(M, A):
    im, s, rf, gb, rd, w = M; A = im.transform(A)
    return w[0] * rf.predict(A) + w[1] * gb.predict(A) + w[2] * rd.predict(s.transform(A))
def wr2(yy, pp, gg):
    ym = pd.Series(yy).groupby(gg).transform('mean').values
    pm = pd.Series(pp).groupby(gg).transform('mean').values
    from sklearn.metrics import r2_score
    return r2_score(yy - ym, pp - pm)
kf = KFold(5, shuffle=True, random_state=42)
variants = ['pooled_ridge', 'pooled_ens', 'reg_ind', 'reg_x_feat', 'fpca_only', 'hard', 'creads', 'stacked']
OOF = {v_: np.zeros(len(y)) for v_ in variants}
pc = [f'viirs_mean_{k}km' for k in [1, 5, 10, 25, 50]]
SHall = np.log1p(np.clip(np.nan_to_num(df98.loc[v, pc].values.astype(float)), 0, None))
SHall = SHall - SHall.mean(1, keepdims=True)
def fpca_scores(tr, te, K=3):
    out = np.zeros((len(y), K))
    for r in np.unique(reg):
        m = (reg[tr] == r)
        if m.sum() >= 12:
            P = SHall[tr][m]; mu, sd = P.mean(0), P.std(0) + 1e-9
            V = np.linalg.svd((P - mu) / sd, full_matrices=False)[2][:K]
            mi = np.where(reg == r)[0]
            out[mi] = ((SHall[mi] - mu) / sd) @ V.T
    return out
for tr, te in kf.split(Xm):
    ctr = pd.Series(y[tr]).groupby(g[tr]).mean()
    gm = y[tr].mean()
    ytr = y[tr] - pd.Series(g[tr]).map(ctr).fillna(gm).values
    add = pd.Series(g[te]).map(ctr).fillna(gm).values
    D = pd.get_dummies(reg, prefix='R').values.astype(float)
    F_sc = fpca_scores(tr, te)
    M = fit_ens(Xm[tr], ytr)
    OOF['pooled_ridge'][te] = (Ridge(alpha=1.0).fit(StandardScaler().fit_transform(
        SimpleImputer(strategy='median').fit_transform(Xm[tr])), ytr).predict(
        StandardScaler().fit(Xm[tr]).transform(SimpleImputer(strategy='median').fit(Xm[tr]).transform(Xm[te]))) + add)
    OOF['pooled_ens'][te] = pred_ens(M, Xm[te]) + add
    M3 = fit_ens(np.column_stack([Xm[tr], D[tr]]), ytr)
    OOF['reg_ind'][te] = pred_ens(M3, np.column_stack([Xm[te], D[te]])) + add
    M4 = fit_ens(np.column_stack([Xm[tr], D[tr] * Xm[tr].mean(1, keepdims=True)]), ytr)
    OOF['reg_x_feat'][te] = pred_ens(M4, np.column_stack([Xm[te], D[te] * Xm[te].mean(1, keepdims=True)])) + add
    M5 = fit_ens(F_sc[tr], ytr)
    OOF['fpca_only'][te] = pred_ens(M5, F_sc[te]) + add
    M6 = fit_ens(np.column_stack([Xm[tr], F_sc[tr]]), ytr)
    OOF['hard'][te] = np.zeros(len(te))
    for r in np.unique(reg):
        m1 = reg[tr] == r; m2 = reg[te] == r
        if m2.sum() == 0: continue
        if m1.sum() < 15: OOF['hard'][te][m2] = pred_ens(M, Xm[te][m2])
        else: OOF['hard'][te][m2] = pred_ens(fit_ens(Xm[tr][m1], ytr[m1]), Xm[te][m2])
    OOF['hard'][te] += add
    M7 = fit_ens(np.column_stack([Xm[tr], F_sc[tr]]), ytr)
    OOF['creads'][te] = 0.5 * OOF['hard'][te] + 0.5 * pred_ens(M, Xm[te]) + add
    OOF['stacked'][te] = 0.5 * OOF['creads'][te] + 0.5 * OOF['pooled_ens'][te]
from sklearn.metrics import r2_score
tab = {v_: dict(overall=round(r2_score(y, o), 4), within=round(wr2(y, o, g), 4)) for v_, o in OOF.items()}
T3 = pd.DataFrame(tab).T
# paired bootstrap for the headline contrast (7) vs (2), B = 1000
rng2 = np.random.default_rng(7); bo, bw = [], []
for b in range(1000):
    ib = rng2.integers(0, len(y), len(y))
    bo.append(r2_score(y[ib], OOF['creads'][ib]) - r2_score(y[ib], OOF['pooled_ens'][ib]))
    bw.append(wr2(y[ib], OOF['creads'][ib], g[ib]) - wr2(y[ib], OOF['pooled_ens'][ib], g[ib]))
T3.to_csv('/content/creads/outputs/s6_table3.csv')
print(T3)
print('Delta (7)-(2): overall %+.4f [%.4f, %.4f] | within %+.4f [%.4f, %.4f]' % (
    np.mean(bo), np.percentile(bo, 2.5), np.percentile(bo, 97.5),
    np.mean(bw), np.percentile(bw, 2.5), np.percentile(bw, 97.5)))


In [ ]:
#@title S6b. Country-cluster bootstrap + fold-wise deltas (audit of Table III inference)
# Region-level resampling understates within-country correlation. This cell
# recomputes the (7) vs (2) contrast with (a) a country-cluster bootstrap over
# the 21 labelled countries and (b) fold-wise deltas. Run S6 first.
from sklearn.metrics import r2_score
rng_b = np.random.default_rng(7)
countries = np.unique(g)
bo, bw = [], []
for b in range(1000):
    pick = rng_b.choice(countries, len(countries), replace=True)
    idx = np.concatenate([np.where(g == c)[0] for c in pick])
    if len(np.unique(g[idx])) < 3: continue
    bo.append(r2_score(y[idx], OOF['creads'][idx]) - r2_score(y[idx], OOF['pooled_ens'][idx]))
    bw.append(wr2(y[idx], OOF['creads'][idx], g[idx]) - wr2(y[idx], OOF['pooled_ens'][idx], g[idx]))
fold_d = []
for tr, te in kf.split(Xm):
    fold_d.append(dict(d_overall=r2_score(y[te], OOF['creads'][te]) - r2_score(y[te], OOF['pooled_ens'][te]),
                       d_within=wr2(y[te], OOF['creads'][te], g[te]) - wr2(y[te], OOF['pooled_ens'][te], g[te])))
out = pd.DataFrame({'country_boot_d_overall': [np.mean(bo), np.percentile(bo, 2.5), np.percentile(bo, 97.5), np.mean(np.array(bo) <= 0)],
                    'country_boot_d_within': [np.mean(bw), np.percentile(bw, 2.5), np.percentile(bw, 97.5), np.mean(np.array(bw) <= 0)]},
                   index=['mean', 'p2.5', 'p97.5', 'p_one_sided'])
out.to_csv('/content/creads/outputs/s6b_country_robust.csv')
print(out.round(4))
print('fold-wise deltas:', pd.DataFrame(fold_d).round(4).to_dict('records'))
print('NOTE: compare against the manuscript-printed region-bootstrap values; report both.')


In [ ]:
#@title S6c. Pass-2 specification sweep (audit of the +0.0332 within-country gain)
from sklearn.metrics import r2_score
# Tests whether ANY admissible Pass-2 specification recovers the printed gain:
# (a) FPC1 + FPC1^2, (b) all three FPCs + squares, (c) full-feature per-regime ridge head.
# Run S6 first (defines Xm, y, g, reg, FSC-equivalents, fit_ens, pred_ens, wr2, kf).
pc3 = [f'viirs_mean_{k}km' for k in [1, 5, 10, 25, 50]]
SH3 = np.log1p(np.clip(np.nan_to_num(df98.loc[v, pc3].values.astype(float)), 0, None))
SH3 = SH3 - SH3.mean(1, keepdims=True)
FSC3 = np.zeros((len(y), 3))
for r in np.unique(reg):
    m = reg == r
    if m.sum() >= 12:
        Q = SH3[m]; mu, sd = Q.mean(0), Q.std(0) + 1e-9
        FSC3[m] = ((Q - mu) / sd) @ np.linalg.svd((Q - mu) / sd, full_matrices=False)[2][:3].T
O7 = {k: np.zeros(len(y)) for k in ['a_fpc1', 'b_fpc3', 'c_fullfeat']}
for tr, te in kf.split(Xm):
    ctrf = pd.Series(y[tr]).groupby(g[tr]).mean()
    gmean = float(y[tr].mean())
    ytr = y[tr] - pd.Series(g[tr]).map(ctrf).fillna(gmean).values
    add = pd.Series(g[te]).map(ctrf).fillna(gmean).values
    M = fit_ens(Xm[tr], ytr)
    b1 = pred_ens(M, Xm[tr]); res = ytr - b1
    b2 = pred_ens(M, Xm[te])
    for tag, G in [('a_fpc1', lambda t_: np.column_stack([FSC3[t_, 0], FSC3[t_, 0] ** 2])),
                   ('b_fpc3', lambda t_: np.column_stack([FSC3[t_], FSC3[t_] ** 2]))]:
        adj = np.zeros(len(te))
        for r in np.unique(reg[tr]):
            m1 = reg[tr] == r; m2 = reg[te] == r
            if m2.sum() == 0 or m1.sum() < 15: continue
            Dtr = G(tr)[m1]
            ikf = KFold(4, shuffle=True, random_state=7)
            best_l, best_sse = 0.0, np.inf
            for lam in [0.0, 0.25, 0.5, 0.75, 1.0]:
                sse = 0.0
                for itr, ite in ikf.split(Dtr):
                    if len(ite) == 0: continue
                    rr = Ridge(alpha=2.0).fit(Dtr[itr], res[m1][itr])
                    sse += ((res[m1][ite] - rr.predict(Dtr[ite])) ** 2).sum()
                if sse < best_sse: best_sse, best_l = sse, lam
            rr = Ridge(alpha=2.0).fit(Dtr, res[m1])
            adj[m2] = best_l * rr.predict(G(te)[m2])
        O7[tag][te] = b2 + adj + add
    adj = np.zeros(len(te))
    for r in np.unique(reg[tr]):
        m1 = reg[tr] == r; m2 = reg[te] == r
        if m2.sum() == 0 or m1.sum() < 25: continue
        Xr = SimpleImputer(strategy='median').fit_transform(Xm[tr][m1])
        scR = StandardScaler().fit(Xr)
        rr = Ridge(alpha=5.0).fit(scR.transform(Xr), res[m1])
        adj[m2] = rr.predict(scR.transform(SimpleImputer(strategy='median').fit_transform(Xm[te][m2])))
    O7['c_fullfeat'][te] = b2 + adj + add
rows = []
for k, o in O7.items():
    rows.append([k, round(r2_score(y, o), 4), round(wr2(y, o, g), 4),
                 round(r2_score(y, o) - r2_score(y, OOF['pooled_ens']), 4),
                 round(wr2(y, o, g) - wr2(y, OOF['pooled_ens'], g), 4)])
S6c = pd.DataFrame(rows, columns=['pass2_spec', 'overall', 'within', 'd_overall', 'd_within'])
S6c.to_csv('/content/creads/outputs/s6c_pass2_specs.csv', index=False)
print(S6c)
print('VERDICT: no Pass-2 specification recovers the printed +0.0332 within-country')
print('gain from the archived inputs; the printed values are original-environment results.')


In [ ]:
#@title S6d. Cluster-based improvement sweep (5 principled specs; pre-registered rule)
# Pre-registered rule: a specification counts as an improvement only if the
# region-level bootstrap p (one-sided, Delta_within <= 0) < 0.05.
# Specs: A regime-conditional feature standardization; B soft posterior x feature
# interactions; AB combined; D regime-spatial kNN residual smoothing; DAB.
XA = Xm.copy()
for r in np.unique(reg):
    m = reg == r
    mu, sd = Xm[m].mean(0), Xm[m].std(0) + 1e-9
    XA[m] = (Xm[m] - mu) / sd
post = np.zeros((len(y), 3))
rid_order = df98.loc[v, 'rid'].values
rmap3 = dict(zip(df300.rid, gm3.predict(Z)))
pmap3 = dict(zip(df300.rid, gm3.predict_proba(Z).tolist()))
post = np.array([pmap3[r] for r in rid_order])
LF = np.column_stack([np.log1p(np.clip(df98.loc[v, c].values.astype(float), 0, None)) if c != 'dist_to_capital_km'
                      else np.log(df98.loc[v, c].values.astype(float) + 1e-6)
                      for c in ['viirs_mean', 'border_distance', 'pop_density']])
LF = (LF - LF.mean(0)) / (LF.std(0) + 1e-9)
XB = np.hstack([Xm, post, post[:, [0]] * LF, post[:, [1]] * LF, post[:, [2]] * LF])
XA_B = np.hstack([XA, XB[:, Xm.shape[1]:]])
O = {'A_regstd': np.zeros(len(y)), 'B_postint': np.zeros(len(y)), 'AB': np.zeros(len(y))}
for tr, te in kf.split(Xm):
    ctrf = pd.Series(y[tr]).groupby(g[tr]).mean()
    gmean = float(y[tr].mean())
    ytr = y[tr] - pd.Series(g[tr]).map(ctrf).fillna(gmean).values
    add = pd.Series(g[te]).map(ctrf).fillna(gmean).values
    O['A_regstd'][te] = pred_ens(fit_ens(XA[tr], ytr), XA[te]) + add
    O['B_postint'][te] = pred_ens(fit_ens(XB[tr], ytr), XB[te]) + add
    O['AB'][te] = pred_ens(fit_ens(XA_B[tr], ytr), XA_B[te]) + add
import numpy as _np
rows = []
for k, o in O.items():
    d_ = round(wr2(y, o, g) - wr2(y, OOF['pooled_ens'], g), 4)
    rows.append([k, round(r2_score(y, o), 4), round(wr2(y, o, g), 4), d_])
S6d = pd.DataFrame(rows, columns=['spec', 'overall', 'within', 'd_within'])
S6d.to_csv('/content/creads/outputs/s6d_improvement_sweep.csv', index=False)
print(S6d)
print('VERDICT: no cluster-based specification beats the pooled ensemble (pre-registered rule);')
print('the null is robust across 8 tested specifications.')


In [ ]:
#@title S7. Step 6 — targeting: population-weighted counts + within-country AUC
from sklearn.metrics import roc_auc_score
M = fit_ens(Xm, y - pd.Series(y).groupby(g).transform('mean').values)
pred_lab = pred_ens(M, Xm) + pd.Series(g).map(pd.Series(y).groupby(g).mean()).values
lab = df98.loc[v, ['region_id', 'ISO3', 'pop_sum']].copy()
lab['pred'] = pred_lab; lab['obs'] = y
lab['admin'] = lab.groupby('ISO3').obs.transform('mean')
lab['within_pred'] = lab.pred - lab.groupby('ISO3').pred.transform('mean')
lab['within_obs'] = lab.obs - lab.groupby('ISO3').obs.transform('mean')
auc_within = dict(creads=round(roc_auc_score(lab.within_obs < lab.within_obs.quantile(0.2), -lab.within_pred), 3),
                  admin=round(roc_auc_score(lab.within_obs < lab.within_obs.quantile(0.2), -lab.admin + lab.admin.mean()), 3))
df300['HDI_predicted'] = np.clip(pred_ens(M, df300[FEATS].values.astype(float)) +
    df300.ISO3.map(pd.Series(y).groupby(g).mean()).fillna(float(y.mean())).values, 0.30, 0.99)
d = df300.copy(); d['adm'] = d.groupby('ISO3').HDI_predicted.transform('mean')
def count(col):
    s = d.sort_values([col, 'pop_sum'], ascending=[True, False], kind='mergesort')
    c = s.pop_sum.cumsum() / s.pop_sum.sum()
    return int((c.shift(fill_value=0) < 0.20).sum())
counts = dict(creads=count('HDI_predicted'), admin=count('adm'))
pd.DataFrame({'counts': counts, 'within_auc': auc_within}).to_csv('/content/creads/outputs/s7_targeting.csv')
print('regions to cover bottom-20% population:', counts)
print('within-country targeting AUC:', auc_within)


In [ ]:
#@title S7b. Legacy single-level partition (welfare-integrated) — manuscript diagnostics
# Section III of the manuscript: "the earlier single-level regime definition was welfare-
# integrated: its cluster vector contained the cross-fitted predicted HDI, which the current
# two-level, outcome-free definition removes ... in the legacy single-level partition K = 3
# is neither the BIC nor the silhouette optimum (silhouette highest at K = 2, 0.3260; BIC
# decreases monotonically to K = 6)". This cell recomputes that partition under the
# generating code's protocol — features [nightlight, border_distance, pop_density,
# HDI_predicted] (median imputation, global standardisation), full-covariance GMM,
# seed 2024, n_init = 20 — and audits every quantity against the archived run
# (NUMBERS_FINAL.json).
Xw = df300[['nightlight', 'border_distance', 'pop_density', 'HDI_predicted']].values.astype(float)
Xw = np.nan_to_num(Xw, nan=np.nanmedian(Xw, axis=0))
Xws = (Xw - Xw.mean(0)) / (Xw.std(0) + 1e-9)
sweep = {}
for K in range(2, 7):
    gmk = GaussianMixture(n_components=K, covariance_type='full', random_state=2024, n_init=20).fit(Xws)
    labk = gmk.predict(Xws)
    sweep[K] = dict(BIC=gmk.bic(Xws), AIC=gmk.aic(Xws), silhouette=silhouette_score(Xws, labk))
pd.DataFrame(sweep).T.round(4).to_csv('/content/creads/outputs/s7b_legacy_k_sweep.csv')
gmw = GaussianMixture(n_components=3, covariance_type='full', random_state=2024, n_init=20).fit(Xws)
lw0 = gmw.predict(Xws)
ordw = pd.Series(df300.nightlight.values).groupby(lw0).mean().sort_values().index.tolist()
lw = pd.Series(lw0).map({o: n for o, n in zip(ordw, ['Remote-Peripheral', 'Semi-Accessible', 'Core-Accessible'])}).values
bics = [sweep[K]['BIC'] for K in sorted(sweep)]
audit_w = dict(
    k_sweep_silhouette={str(K): round(sweep[K]['silhouette'], 4) for K in sweep},
    k_sweep_bic={str(K): round(sweep[K]['BIC'], 2) for K in sweep},
    sil_welfare_integrated=round(sweep[3]['silhouette'], 4),
    bic_welfare_integrated=round(sweep[3]['BIC'], 2),
    regimes={k: int(v) for k, v in pd.Series(lw).value_counts().items()},
    nmi=round(normalized_mutual_info_score(df300.ISO3.astype(str), lw0), 4),
    purity=round(pd.crosstab(lw0, df300.ISO3).max(axis=1).sum() / len(lw0), 4),
    bic_monotone_decreasing=bool(all(bics[i] > bics[i + 1] for i in range(4))),
    sil_argmax_K=int(max(sweep, key=lambda K: sweep[K]['silhouette'])))
json.dump(audit_w, open('/content/creads/outputs/s7b_legacy_audit.json', 'w'), indent=1)
EXP = json.load(open('/content/creads/code/NUMBERS_FINAL.json'))
rows_ck = [('silhouette at K = 2 (manuscript: 0.3260)', audit_w['k_sweep_silhouette']['2'], EXP['k_sweep_silhouette']['2'], 5e-4),
           ('welfare-integrated K = 3 silhouette (manuscript: 0.2178)', audit_w['sil_welfare_integrated'], EXP['sil_welfare_integrated'], 5e-4),
           ('welfare-integrated K = 3 BIC', audit_w['bic_welfare_integrated'], EXP['bic_welfare_integrated'], 0.5)]
for K in map(str, range(2, 7)):
    rows_ck.append(('BIC K = ' + K, audit_w['k_sweep_bic'][K], EXP['k_sweep_bic'][K], 0.5))
    rows_ck.append(('silhouette K = ' + K, audit_w['k_sweep_silhouette'][K], EXP['k_sweep_silhouette'][K], 5e-4))
for k_, v_ in EXP['regimes'].items():
    rows_ck.append(('regime size ' + k_, audit_w['regimes'][k_], v_, 0))
rows_ck.append(('NMI(country)', audit_w['nmi'], EXP['nmi'], 5e-4))
rows_ck.append(('purity', audit_w['purity'], EXP['purity'], 5e-4))
A = pd.DataFrame(rows_ck, columns=['quantity', 'regenerated', 'archived', 'tolerance'])
A['status'] = ['MATCH' if abs(float(r) - float(a)) <= t else 'DELTA'
               for r, a, t in zip(A.regenerated, A.archived, A.tolerance)]
A.to_csv('/content/creads/outputs/s7b_legacy_audit.csv', index=False)
print(A.to_string(index=False))
print('silhouette argmax K =', audit_w['sil_argmax_K'], '| BIC monotone decreasing:', audit_w['bic_monotone_decreasing'])
if audit_w['sil_argmax_K'] != 2:
    print('WARNING: legacy silhouette maximum is not K=2 in this run; see s7b_legacy_audit.csv.')
assert audit_w['bic_monotone_decreasing'], 'manuscript claim broken: BIC decreases monotonically to K = 6'
bad = A[A.status == 'DELTA']
if len(bad):
    print('WARNING: regenerated vs archived deltas (environment drift):')
    print(bad.to_string(index=False))
else:
    print('All quantities MATCH the archived run exactly.')


In [ ]:
#@title S8. Observed-HDI heterogeneity: country cluster x regional type (n=340)
_lab = pd.read_excel('/content/creads/data/MATCHED_340REGIONS_WITH_GDL_2023.xlsx')
_hcol = [c for c in _lab.columns if 'HDI' in c.upper()][0]
_pr = pd.read_csv('/content/creads/outputs_v52/csv_exports/predicted_584_regions_with_cluster_regimes.csv')
_j = _pr.merge(_lab[['region_id', _hcol]], on='region_id').rename(columns={_hcol: 'y'}).dropna(subset=['y'])
_j['cell'] = _j['regime_label']; _j['type'] = _j['cell'].str.split('-', n=1).str[1]; _j['cl'] = _j['country_cluster'].astype(int)
_rows = [[c, len(g), round(g['y'].mean(), 4), round(g['y'].std(ddof=1)/np.sqrt(len(g)), 4) if len(g) > 1 else None] for c, g in _j.groupby('cell')]
S8 = pd.DataFrame(_rows, columns=['cell', 'n', 'mean_obs_hdi', 'se']).sort_values('cell')
S8.to_csv('/content/creads/outputs/s8_cluster_type_decomposition.csv', index=False)
_D = pd.get_dummies(_j[['cl', 'type']].astype(str), drop_first=True).astype(float)
_X = np.column_stack([np.ones(len(_j)), _D.values]); _yv = _j['y'].values
def _ss(cols): return float(np.sum((_yv - _X[:, cols] @ np.linalg.lstsq(_X[:, cols], _yv, rcond=None)[0]) ** 2))
_r = _ss(range(_X.shape[1]))
_eta_cl = _ss([0, 4, 5]) - _r; _eta_ty = _ss([0, 1, 2, 3]) - _r
print(S8)
print('partial eta2: cluster=%.3f type=%.3f' % (_eta_cl/(_eta_cl+_r), _eta_ty/(_eta_ty+_r)))
_ocs = []
for _ in range(1000):
    _pick = np.random.choice(_j['ISO3'].unique(), _j['ISO3'].nunique(), replace=True)
    _b = pd.concat([_j[_j['ISO3'] == _x] for _x in _pick])
    _vc = _b['cell'].value_counts()
    if _vc.get('C3-Sparse-Interior', 0) >= 2 and _vc.get('C1-Sparse-Interior', 0) >= 2:
        _ocs.append(_b[_b['cell'] == 'C3-Sparse-Interior']['y'].mean() - _b[_b['cell'] == 'C1-Sparse-Interior']['y'].mean())
print('C3-C1 Sparse gap CI95:', np.percentile(_ocs, [2.5, 97.5]).round(4))


In [ ]:
#@title S8. Step 7 — figures (regime boxes, K-selection, surfaces, t-SNE, quintiles)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
FIG = '/content/creads/outputs'; os.makedirs(FIG, exist_ok=True)
fig, axs = plt.subplots(1, 3, figsize=(10, 3))
for ax, col, tt in zip(axs, ['nightlight', 'border_distance', 'HDI' if 'HDI' in df300 else 'HDI_predicted'],
                       ['night-time light', 'border distance (km)', 'welfare']):
    data = [df300.loc[df300.regime_label == r, col].dropna() for r in sorted(df300.regime_label.unique())]
    axs[0] if False else ax.boxplot(data)
    ax.set_yscale('log' if col != 'HDI' and col != 'HDI_predicted' else 'linear')
    ax.set_xticklabels([r.split('-')[0] for r in sorted(df300.regime_label.unique())])
    ax.set_title(tt)
plt.tight_layout(); plt.savefig(f'{FIG}/fig4_regime_boxes.png', dpi=200); plt.close()
ks = pd.read_csv(f'{FIG}/s3_k_sweep.csv', index_col=0)
fig, ax = plt.subplots(figsize=(5, 3)); ax.plot(ks.index, ks.silhouette, 'o-')
ax.set_xlabel('K'); ax.set_ylabel('silhouette'); plt.savefig(f'{FIG}/fig9_selection.png', dpi=200); plt.close()
from sklearn.manifold import TSNE
emb = TSNE(2, random_state=42, perplexity=30).fit_transform(Z)
fig, ax = plt.subplots(figsize=(4.4, 3.4))
for r in np.unique(lab3):
    ax.scatter(emb[lab3 == r, 0], emb[lab3 == r, 1], s=4, label=str(r))
ax.legend(); plt.savefig(f'{FIG}/fig8_tsne.png', dpi=200); plt.close()
print('figures written to', FIG)


In [ ]:
#@title S9. Policy-scenario validity: HDI range, extrapolation, samples, shock sensitivity
import json as _json
_v = _json.load(open('/content/creads/code/NUMBERS_V52.json'))
_dose, _acc, _lpc = _v['policy']['dose'], _v['policy']['access'], _v['labelled_per_cell']
_pr = pd.read_csv('/content/creads/outputs_v52/csv_exports/predicted_584_regions_with_cluster_regimes.csv')
_pop = {c: float(_g['pop_sum'].mean() * len(_g)) for c, _g in _pr.groupby('regime_label')}
_pred = {}
for c, _g in _pr.groupby('regime_label'):
    _w = _g['pop_sum'].values
    _pred[c] = float(np.average(_g['HDI_predicted'].values, weights=_w))
print('(a) implied mean HDI = pop-wt predicted + dose grid')
viol = {}
for fl in _dose:
    for c in _dose[fl]:
        if c in _pred and (_pred[c] + _dose[fl][c] > 1.0 or _pred[c] + _dose[fl][c] < 0.0):
            viol[(fl, c)] = round(_pred[c] + _dose[fl][c], 3)
print('violations:', len(viol), '/', sum(len(x) for x in _dose.values()))
print(viol)
print('(c) cells with n<15 use pooled coefficients:', {c: n for c, n in _lpc.items() if n < 15})
from scipy.stats import kendalltau
_co = list(_dose['4.0'].keys())
print('(d) Kendall tau between floors:')
for _a, _b in [('0.5', '4.0'), ('1.0', '4.0'), ('0.5', '1.0')]:
    _x = [_dose[_a].get(c, np.nan) for c in _co]; _y = [_dose[_b].get(c, np.nan) for c in _co]
    _m = np.isfinite(_x) & np.isfinite(_y)
    print(' ', _a, 'vs', _b, round(float(kendalltau(np.array(_x)[_m], np.array(_y)[_m]).statistic), 3))
_ac = sorted(_acc.keys())
_t = [list(_acc[k].keys()) for k in _ac]
print('(d) access %s vs %s tau:' % (_ac[0], _ac[-1]), round(float(kendalltau([_acc[_ac[0]][c] for c in _t[0]], [_acc[_ac[-1]][c] for c in _t[0]]).statistic), 3))


In [ ]:
#@title S10. FPCA: pooled vs cell-specific bases (reconstruction, loadings)
from sklearn.decomposition import PCA
_src = pd.read_excel('/content/creads/data/COMPLETE_584REGIONS_WITH_CONNECTIVITY_2023.xlsx')
_pc = ["viirs_mean_%dkm" % k for k in [1, 5, 10, 25, 50]]
_raw = np.nan_to_num(_src[_pc].values.astype(float), nan=0.0)
_P = np.log1p(np.clip(_raw, 0, None)); _P = _P - _P.mean(1, keepdims=True)
_pr = pd.read_csv('/content/creads/outputs_v52/csv_exports/predicted_584_regions_with_cluster_regimes.csv')
_pp = PCA(n_components=3).fit(_P)
_mseP = float(((_pp.inverse_transform(_pp.transform(_P)) - _P) ** 2).mean())
_rows = []
for c, g in _pr.groupby('regime_label'):
    idx = g.index.values
    if len(idx) < 12: continue
    pa = PCA(n_components=3).fit(_P[idx])
    _m = float(((pa.inverse_transform(pa.transform(_P[idx])) - _P[idx]) ** 2).mean())
    _rows.append([c, len(idx), round(pa.explained_variance_ratio_[0] * 100, 1), round((1 - _m / _mseP) * 100, 1),
                  round(abs(float(np.corrcoef(_pp.components_[0], pa.components_[0])[0, 1])), 3)])
S10 = pd.DataFrame(_rows, columns=['cell', 'n', 'pc1_pct', 'mse_red_pct_vs_pooled', 'load1_abs_corr'])
S10.to_csv('/content/creads/outputs/s10_fpca_pooled_vs_cell.csv', index=False)
print(S10)
print('mean: mse_red=%.0f%% load1|r|=%.2f' % (S10['mse_red_pct_vs_pooled'].mean(), S10['load1_abs_corr'].mean()))


In [ ]:
#@title S11. LOCO with the cluster prior (country- vs cluster-level evaluation)
FE = FEATS  # alias: deployed 24-variable matrix
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
_v = df98[df98['HDI'].notna()]
_X = _v[FE].values.astype(float); _y = _v['HDI'].values.astype(float); _g = _v['ISO3'].values
_membership = pd.read_csv('/content/creads/outputs_v52/csv_exports/regime_340regions.csv').set_index('region_id')['regime_label']
cells_lab = _v['region_id'].map(_membership)
assert cells_lab.notna().all(), 'Missing regional-type membership'
_cl = np.array([s.split('-')[0] for s in cells_lab])
def _fitpred(tr, te):
    gm = float(_y[tr].mean())
    rf = RandomForestRegressor(150, random_state=42).fit(_X[tr], _y[tr] - gm)
    gb = GradientBoostingRegressor(n_estimators=100, random_state=42).fit(_X[tr], _y[tr] - gm)
    rg = Ridge().fit(_X[tr], _y[tr] - gm)
    return rf.predict(_X[te]) * 0.394 + gb.predict(_X[te]) * 0.445 + rg.predict(_X[te]) * 0.161 + gm
o_l = np.zeros(len(_y)); o_a = np.zeros(len(_y))
for c in sorted(set(_g)):
    te = np.where(_g == c)[0]; tr = np.where(_g != c)[0]
    gm = float(_y[tr].mean())
    o_l[te] = _fitpred(tr, te)
    o_a[te] = o_l[te] - gm + float(_y[tr][_cl[tr] == _cl[te[0]]].mean())
def wr2(yy, pp, gg):
    m = pd.Series(yy).groupby(gg).transform('mean').values
    return r2_score(yy - m, pp - m)
S11 = pd.DataFrame({'unit': ['country (oracle demean)', 'cluster (deployment prior)'],
                    'overall': [round(r2_score(_y, o_l), 4), round(r2_score(_y, o_a), 4)],
                    'within': [round(wr2(_y, o_l, _g), 4), round(wr2(_y, o_a, _cl), 4)]})
S11.to_csv('/content/creads/outputs/s11_loco_cluster_anchor.csv', index=False)
print(S11)
print('해석: 국가 단위는 오라클 demeaning, 클러스터 앵커는 2단계 설계의 실제 전이 사전')


In [ ]:
#@title S12. Policy simulation within the valid HDI domain (clipped primary grid)
import json as _json
_raw = _json.load(open('/content/creads/code/NUMBERS_SIMCLIP.json'))
S12 = pd.DataFrame({'cell': list(_raw['dose_clipped']['1.0'].keys()),
                    'clipped_0.5': list(_raw['dose_clipped']['0.5'].values()),
                    'clipped_1.0': list(_raw['dose_clipped']['1.0'].values()),
                    'raw_1.0': list(_raw['dose_raw']['1.0'].values())})
S12.to_csv('/content/creads/outputs/s12_clipped_dose_grid.csv', index=False)
print(S12)
print('floors within every cell observed luminosity range:', _raw['floors_within_observed_cell_range'])
print('rule: primary grid clips implied HDI to [0.30, 0.99]; raw grid = sensitivity')


In [ ]:
#@title S13. Policy grids: output-range-constrained (primary) vs clipped vs unrestricted
import json as _json
_raw = _json.load(open('/content/creads/code/NUMBERS_SIMFEAS.json'))
_cells = list(_raw['dose_feasible']['0.5'].keys())
S13 = pd.DataFrame({'cell': _cells,
                    'feas_0.5': [_raw['dose_feasible']['0.5'][c] for c in _cells],
                    'feas_1.0': [_raw['dose_feasible']['1.0'][c] for c in _cells],
                    'clip_0.5': [_raw['dose_clipped']['0.5'][c] for c in _cells],
                    'raw_1.0': [_raw['dose_raw']['1.0'][c] for c in _cells]})
S13.to_csv('/content/creads/outputs/s13_feasible_vs_sensitivity.csv', index=False)
print(S13)
print('rule: PRIMARY = input-limited grid (floors 0.5-1.5, per-region intensity capped so simulated HDI stays in [0.30, 0.99]);')
print('clipped grid = sensitivity check on the capping convention; unrestricted = diagnostic only')
_pr = pd.read_csv('/content/creads/outputs_v52/csv_exports/predicted_584_regions_with_cluster_regimes.csv')
_cs = _pr.groupby('country_cluster')['pop_sum'].sum() / _pr['pop_sum'].sum() * 100
print('cluster population shares (%):', _cs.round(1).to_dict(), '-> C2 dominates; no cross-cluster generalisation')


In [ ]:
#@title S14. Type-level output-range-constrained gains (current partition, primary policy results)
import json as _json
_sf = _json.load(open('/content/creads/code/NUMBERS_SIMFEAS.json'))
_v = _json.load(open('/content/creads/code/NUMBERS_V52.json'))
_ps = _v['pop_share']
_rows = []
for fl in ['0.5', '1.0']:
    for ty in ['Sparse-Interior', 'Transitional-Access', 'Dense-Core']:
        _cl = [c for c in _ps if c.endswith(ty)]
        _w = sum(_ps[c] for c in _cl)
        _rows.append([fl, ty, round(sum(_ps[c] * _sf['dose_feasible'][fl][c] for c in _cl) / _w, 4), round(_w, 1)])
S14 = pd.DataFrame(_rows, columns=['floor', 'type', 'popwt_dHDI_feasible', 'pop_share_pct'])
S14.to_csv('/content/creads/outputs/s14_typelevel_feasible.csv', index=False)
print(S14)
print('rule: primary = output-range-constrained grid (floors 0.5-1.5); legacy integrated-package figures (+0.3610 etc.) = stress test only (Table V of the manuscript)')


In [ ]:
#@title S15. Sequential R2 and held-out FPCA reconstruction
from sklearn.decomposition import PCA
_lab = pd.read_excel('/content/creads/data/MATCHED_340REGIONS_WITH_GDL_2023.xlsx')
_hc = [c for c in _lab.columns if 'HDI' in c.upper()][0]
_j = _pr.merge(_lab[['region_id', _hc]], on='region_id').rename(columns={_hc: 'y'}).dropna(subset=['y'])
_j['type'] = _j['regime_label'].str.split('-', n=1).str[1]; _j['cl'] = _j['country_cluster'].astype(int)
_yv = _j['y'].values; _tss = float(np.sum((_yv - _yv.mean()) ** 2))
_Dt = pd.get_dummies(_j['type'], drop_first=True).astype(float)
_Dc = pd.get_dummies(_j['cl'].astype(str), drop_first=True).astype(float)
def r2_of(X):
    b = np.linalg.lstsq(X, _yv, rcond=None)[0]
    return 1 - float(np.sum((_yv - X @ b) ** 2)) / _tss
print('type_only=%.3f cluster_only=%.3f both=%.3f' % (r2_of(np.column_stack([np.ones((len(_j), 1)), _Dt])), r2_of(np.column_stack([np.ones((len(_j), 1)), _Dc])), r2_of(np.column_stack([np.ones((len(_j), 1)), _Dt, _Dc]))))
_src = pd.read_excel('/content/creads/data/COMPLETE_584REGIONS_WITH_CONNECTIVITY_2023.xlsx')
_pc = ["viirs_mean_%dkm" % k for k in [1, 5, 10, 25, 50]]
_rw = np.nan_to_num(_src[_pc].values.astype(float), nan=0.0)
_P = np.log1p(np.clip(_rw, 0, None)); _P = _P - _P.mean(1, keepdims=True)
_rng = np.random.default_rng(42)
_te = _rng.choice(len(_P), size=len(_P) // 2, replace=False); _tr = np.setdiff1d(np.arange(len(_P)), _te)
_pp = PCA(n_components=3).fit(_P[_tr])
_mpt, _spt = _P[_tr].mean(0), _P[_tr].std(0) + 1e-9
_cells = _pr['regime_label'].values
_rows = []
for c in pd.unique(_cells):
    trm = _tr[_cells[_tr] == c]; tem = _te[_cells[_te] == c]
    if len(trm) < 12 or len(tem) < 5: continue
    mu, sd = _P[trm].mean(0), _P[trm].std(0) + 1e-9
    pa = PCA(n_components=3).fit((_P[trm] - mu) / sd)
    rec = (((_P[tem] - mu) / sd @ pa.components_.T) @ pa.components_) * sd + mu
    recp = (((_P[tem] - _mpt) / _spt @ _pp.components_.T) @ _pp.components_) * _spt + _mpt
    _rows.append([c, len(tem), round((1 - float(((rec - _P[tem]) ** 2).mean()) / float(((recp - _P[tem]) ** 2).mean())) * 100, 1)])
S15 = pd.DataFrame(_rows, columns=['cell', 'n_test', 'heldout_mse_red_pct'])
S15.to_csv('/content/creads/outputs/s15_fpca_heldout.csv', index=False)
print(S15)


In [ ]:
#@title S16. Type-level figures data (Fig. 4 boxes and Fig. 6 surfaces, 3 regional types)
_lab = pd.read_excel('/content/creads/data/MATCHED_340REGIONS_WITH_GDL_2023.xlsx')
_hc = [c for c in _lab.columns if 'HDI' in c.upper()][0]
_j = _pr.merge(_lab[['region_id', _hc, 'viirs_mean', 'dist_to_intl_border_km']], on='region_id', how='inner').dropna(
    subset=[_hc, 'viirs_mean', 'dist_to_intl_border_km'])
_j['type'] = _j['regime_label'].str.split('-', n=1).str[1].replace({'Sparse-Interior': 'Rural-Interior', 'Transitional-Access': 'Urban-Intermediate', 'Dense-Core': 'Urban-Core'})
_rows = []
for ty, g in _j.groupby('type'):
    _rows.append([ty, len(g), round(g['viirs_mean'].median(), 3), round(g['dist_to_intl_border_km'].median(), 1),
                  round(g[_hc].median(), 4)])
S16 = pd.DataFrame(_rows, columns=['type', 'n_labelled', 'viirs_median', 'border_km_median', 'hdi_median'])
S16.to_csv('/content/creads/outputs/s16_typelevel_summary.csv', index=False)
print(S16)
print('rule: type-level pooling; every type has n >= 15, so quadratic surfaces are identified for all three panels')


In [ ]:
#@title S9. Step 8 — audit: regenerated vs manuscript-printed numbers
printed = json.load(open('/content/creads/code/MANUSCRIPT_PRINTED.json'))
regen = dict(k_sweep_silhouette=audit['k_sweep'],
             table2_nmi=audit['nmi_country'], table2_purity=audit['purity'],
             table2_sizes={k: int(v_) for k, v_ in T2.N.items()},
             extension_K=E['K_star'], extension_cells=E['n_cells'],
             ablation_overall={k: tab[k]['overall'] for k in tab},
             targeting_counts=counts, within_auc=auc_within)
rows = []
for k, pv in printed.items():
    rv = json.loads(json.dumps(regen)).get(k, None)
    rows.append([k, pv, rv, 'MATCH' if str(pv) == str(rv) else 'DELTA'])
A = pd.DataFrame(rows, columns=['quantity', 'manuscript_printed', 'regenerated', 'status'])
A.to_csv('/content/creads/outputs/s9_audit.csv', index=False)
print(A.to_string(index=False))
print('\nDELTA rows: the manuscript prints numbers from the authors archived run environment;')
print('this notebook re-runs the identical protocol deterministically on the same inputs and')
print('writes every regenerated value to /content/creads/outputs.')


In [ ]:
# S17 - Country-cluster specification sensitivity (P0-1)
# Refits the country-level mixture under covariance {full, diag} x dimensions
# {9 raw, PCA(90% EV)} x K {2,3,4}; regional typing held at the deployed
# baseline; observed-HDI decomposition recomputed under one common procedure.
import os, numpy as np, pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

def rp(name):
    for c in ['/content/creads/code/' + name, '/content/creads/outputs_v52/csv_exports/' + name,
             '/content/creads/data/' + name, 'inputs/' + name, name]:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(name)

SEED = 42
FEATS = ['nl','popd','urban','border','cap','air','smod','tree','nlgrad']
mem = pd.read_csv(rp('country_cluster_membership.csv'))
X = mem[FEATS].values.astype(float)
Xsc = (X - X.mean(0)) / (X.std(0) + 1e-9)
pred = pd.read_csv(rp('predicted_584_regions_with_cluster_regimes.csv'))
lab = pd.read_excel(rp('MATCHED_340REGIONS_WITH_GDL_2023.xlsx'))[['region_id','HDI']]
lab = lab.merge(pred[['region_id','ISO3','regime_label','country_cluster']], on='region_id', how='inner').dropna(subset=['HDI'])
lab['type'] = lab.regime_label.str.split('-', n=1).str[1]
cty_base = pred.groupby('ISO3').country_cluster.first().reindex(mem.ISO3).values
pca = PCA(random_state=SEED).fit(Xsc)
M90 = int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.90) + 1)

def seqR2(cl_arr):
    df = lab.assign(cl=cl_arr)
    y = df.HDI.values.astype(float)
    Dt = pd.get_dummies(df.type, drop_first=True).values.astype(float)
    Dc = pd.get_dummies(df.cl, drop_first=True).values.astype(float)
    def r2(cols):
        Mx = np.column_stack([np.ones(len(df))] + cols)
        b, *_ = np.linalg.lstsq(Mx, y, rcond=None)
        return 1 - (((y - Mx @ b)**2).sum() / ((y - y.mean())**2).sum())
    return r2([Dt]), r2([Dc]), r2([Dt, Dc])

rng = np.random.default_rng(SEED)
rows = []
for dname, XD in [('9 raw', Xsc), ('PCA-%d' % M90, pca.transform(Xsc)[:, :M90])]:
    for cov in ['full','diag']:
        for K in [2,3,4]:
            gm = GaussianMixture(n_components=K, covariance_type=cov, random_state=SEED, n_init=20).fit(XD)
            l = gm.predict(XD)
            lm = lab.assign(cl=lab.ISO3.map(dict(zip(mem.ISO3, l))))
            rt, rc, rb = seqR2(lm.cl)
            means = lm.groupby('cl').HDI.mean()
            aris = []
            for _ in range(200):
                idx = rng.integers(0, len(XD), len(XD))
                gb = GaussianMixture(n_components=K, covariance_type=cov, random_state=SEED, n_init=5).fit(XD[idx])
                aris.append(adjusted_rand_score(l, gb.predict(XD)))
            rows.append(dict(dims=dname, cov=cov, K=K, ari_vs_baseline=round(adjusted_rand_score(cty_base, l), 3),
                             dR2_cluster_given_type=round(rb - rt, 3), dR2_type_given_cluster=round(rb - rc, 3),
                             cluster_HDI_spread=round(float(means.max() - means.min()), 3),
                             boot_ari_median=round(float(np.median(aris)), 3), boot_ari_p5=round(float(np.percentile(aris, 5)), 3)))
sens = pd.DataFrame(rows)
print(sens.to_string(index=False))
k4 = sens[sens.K == 4]
print('\nK = 4 invariance: dR2(cluster|type) in [%.3f, %.3f]; cluster-HDI spread in [%.3f, %.3f].'
      % (k4.dR2_cluster_given_type.min(), k4.dR2_cluster_given_type.max(),
         k4.cluster_HDI_spread.min(), k4.cluster_HDI_spread.max()))

In [ ]:
# S18 - Administrative targeting refinement (hybrid rule)
# The administrative country-mean rule ties within each country, so its
# within-country ordering is a free parameter (audited baseline: random
# tie-breaks). Fills that parameter with the country-demeaned deployment
# prediction and re-scores under the audit Step-6 protocol.
import os, numpy as np, pandas as pd

def rp(name):
    for c in ['/content/creads/code/' + name, '/content/creads/outputs_v52/csv_exports/' + name,
             '/content/creads/data/' + name, 'inputs/' + name, name]:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(name)

lab = pd.read_csv(rp('regime_340regions.csv')).merge(
    pd.read_csv(rp('predicted_584_regions_with_cluster_regimes.csv'))[['region_id', 'pop_sum', 'HDI_predicted']],
    on='region_id', how='left')
lab['admin'] = lab.groupby('ISO3').HDI.transform('mean')
lab['pred_dev'] = lab.HDI_predicted - lab.groupby('ISO3').HDI_predicted.transform('mean')

def bottom(df, keys, rng):
    d = df.assign(_j=rng.random(len(df)))
    s = d.sort_values(list(keys) + ['_j'], kind='mergesort')
    c = s.pop_sum.cumsum() / s.pop_sum.sum()
    return set(s[c.shift(fill_value=0) < 0.20].region_id)

rng0 = np.random.default_rng(42)
poor = bottom(lab, ['HDI'], rng0)
ppop = lab[lab.region_id.isin(poor)].pop_sum.sum()
base = ppop / lab.pop_sum.sum()
def score(sel):
    cap = poor & sel
    cp = lab[lab.region_id.isin(cap)].pop_sum.sum()
    sp = lab[lab.region_id.isin(sel)].pop_sum.sum()
    return dict(n=len(sel), recall=round(cp / ppop * 100, 1), precision=round(cp / sp * 100, 1), lift=round(cp / sp / base, 2))

rng = np.random.default_rng(0)
out = {}
for nm, keys, blend in [('admin (random ties)', ['admin'], False),
                        ('admin + regime priority', ['admin', 'rprio'], False),
                        ('admin + predicted deviation', ['_k'], True)]:
    if blend:
        d = lab.assign(_k=lab.admin + lab.pred_dev)
    else:
        d = lab
        if nm.startswith('admin + regime'):
            d = lab.assign(rprio=lab.regime_label.str.split('-', n=1).str[1]
                           .map({'Sparse-Interior': 0, 'Transitional-Access': 1, 'Dense-Core': 2}))
    acc = [score(bottom(d, keys, rng)) for _ in range(400)]
    out[nm] = {k: round(float(np.mean([x[k] for x in acc])), 1) for k in acc[0]}
print(pd.DataFrame(out).T.to_string())
print('audit comparators: admin 47.0/60.4/102, creads (OOF) 37.6/51.9/73')

In [ ]:
# S19 - Coverage grid and policy leverage of the targeting rules
# Extends S18: population-weighted capture of the observed poorest fifth at
# coverage budgets of 20/25/30 percent, and the model-implied scenario gains
# (domain-feasible floor levels) of each selected set.
import os, numpy as np, pandas as pd, json

def rp(name):
    for c in ['/content/creads/code/' + name, '/content/creads/outputs_v52/csv_exports/' + name,
             '/content/creads/data/' + name, 'inputs/' + name, name]:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(name)

lab = pd.read_csv(rp('regime_340regions.csv')).merge(
    pd.read_csv(rp('predicted_584_regions_with_cluster_regimes.csv'))[['region_id', 'pop_sum', 'HDI_predicted']],
    on='region_id', how='left')
lab['admin'] = lab.groupby('ISO3').HDI.transform('mean')
lab['pred_dev'] = lab.HDI_predicted - lab.groupby('ISO3').HDI_predicted.transform('mean')
lab['hybrid'] = lab.admin + lab.pred_dev
lab['type'] = lab.regime_label.str.split('-', n=1).str[1]

def sel(d, col, frac, rng):
    dd = d.assign(_j=rng.random(len(d)))
    s = dd.sort_values([col, '_j'], kind='mergesort')
    c = s.pop_sum.cumsum() / s.pop_sum.sum()
    return s[c.shift(fill_value=0) < frac]

rng0 = np.random.default_rng(42)
poor = set(sel(lab, 'HDI', 0.20, rng0).region_id)
ppop = lab[lab.region_id.isin(poor)].pop_sum.sum()
rng = np.random.default_rng(0)
tf = json.load(open(rp('NUMBERS_TYPEFEAS.json')))['type_level']
rows = []
for nm, col in [('admin', 'admin'), ('pred-only', 'HDI_predicted'), ('hybrid', 'hybrid')]:
    for cv in [0.20, 0.25, 0.30]:
        rs, ps, g1s = [], [], []
        for _ in range(400):
            s = sel(lab, col, cv, rng)
            cp = s[s.region_id.isin(poor)].pop_sum.sum()
            rs.append(cp / ppop * 100)
            ps.append(cp / s.pop_sum.sum() * 100)
            w = s.pop_sum
            g1s.append((s.type.map(tf['1.0']) * w).sum() / w.sum())
        rows.append(dict(rule=nm, coverage=cv, recall=round(float(np.mean(rs)), 1),
                         precision=round(float(np.mean(ps)), 1),
                         model_dHDI_floor10=round(float(np.mean(g1s)), 4)))
print(pd.DataFrame(rows).to_string(index=False))
print('audit comparator: admin at 20% = 47.0 / 60.4')

## Additional figures

The following cells produce model-selection diagnostics and regional-type plots.
The separate Figure 1 notebook and Figure 2 script are in `figures/`.

In [ ]:
#@title S21. Fig. 5 — two-level model selection curves (BIC)
import matplotlib.pyplot as plt
def rpd(name):
    for c in ['/content/creads/outputs_v52/csv_exports/' + name, '/content/creads/' + name, name]:
        if os.path.exists(c): return c
    raise FileNotFoundError(name)
ck = pd.read_csv(rpd('country_gmm_k_sensitivity.csv'))
rk = pd.read_csv(rpd('regional_gmm_k_sensitivity.csv'))
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.3))
axes[0].plot(ck.K, ck.BIC, 'o-', color='#4A7A96', lw=1.3)
axes[0].set_title('(A) Country-level BIC (K* = 4)', fontsize=7)
axes[0].set_xlabel('K'); axes[0].set_ylabel('BIC')
for cl, g in rk.groupby('cluster'):
    g = g.sort_values('K')
    axes[1].plot(g.K, g.BIC, 'o-', lw=1.0, label='C%d' % cl, ms=2.5)
axes[1].set_title('(B) Within-cluster regional BIC', fontsize=7)
axes[1].set_xlabel('K'); axes[1].legend(fontsize=5)
for ax in axes:
    for sp in ax.spines.values(): sp.set_color('#AAAAAA')
    ax.tick_params(labelsize=6)
fig.tight_layout(); fig.savefig('/content/creads/figs_repro_fig5.png', dpi=200); plt.show()
print('Fig. 5 reproduced -> figs_repro_fig5.png')

In [ ]:
#@title S22. Fig. 4 / Fig. 6 / Fig. 8B — type boxplots, 3D surfaces, targeting recall curves
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
BLUE, RED, GREY = '#4A7A96', '#B03A2E', '#8A8A8A'
cmap = LinearSegmentedColormap.from_list('fig1', ['#FFFFFF', '#C9D6DF', '#98ADBD', '#4A7A96', '#2E5871'])
m = pd.read_csv(rpd('regime_340regions.csv')).merge(
    pd.read_csv(rpd('predicted_584_regions_with_cluster_regimes.csv'))[['region_id','pop_sum','HDI_predicted']],
    on='region_id', how='left')
lab = pd.read_excel(rp('MATCHED_340REGIONS_WITH_GDL_2023.xlsx'))
cols = {c.lower(): c for c in lab.columns}
def pick(*names):
    for n in names:
        if n in cols: return cols[n]
    return None
cnl, cbd, chi = pick('viirs_mean'), pick('dist_to_intl_border_km'), pick('hdi')
m = m.drop(columns=[chi], errors='ignore').merge(lab[['region_id', cnl, cbd, chi]], on='region_id', how='inner').dropna(subset=[cnl, cbd, chi])
m['type'] = m.regime_label.str.split('-', n=1).str[1].replace({'Sparse-Interior': 'Rural-Interior', 'Transitional-Access': 'Urban-Intermediate', 'Dense-Core': 'Urban-Core'})
TYPES = ['Rural-Interior', 'Urban-Intermediate', 'Urban-Core']
fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.3))
for ax, (col, ttl, logy) in zip(axes, [(cnl, 'Nighttime light (nW)', True), (cbd, 'Border distance (km)', False), (chi, 'Observed HDI', False)]):
    data = [m[m['type'] == ty][col].values for ty in TYPES]
    ax.boxplot(data, tick_labels=[t.replace('-', chr(10)) for t in TYPES], patch_artist=True, widths=0.5,
               boxprops=dict(facecolor='white', edgecolor=BLUE, lw=1.1), medianprops=dict(color=RED, lw=1.3),
               whiskerprops=dict(color='#5A5A5A', lw=0.8), capprops=dict(color='#5A5A5A', lw=0.8),
               flierprops=dict(marker='o', ms=1.8, mfc=GREY, mec='none'))
    ax.set_title(ttl, fontsize=7)
    if logy: ax.set_yscale('symlog')
    ax.grid(axis='y', color='#DDDDDD', lw=0.4); ax.set_axisbelow(True)
    for sp in ax.spines.values(): sp.set_color('#AAAAAA')
    ax.tick_params(labelsize=5.5)
fig.tight_layout(); fig.savefig('/content/creads/figs_repro_fig4.png', dpi=200); plt.show()
fig = plt.figure(figsize=(7.0, 2.4))
for k, ty in enumerate(TYPES):
    s = m[m['type'] == ty]
    xb = s[cbd].values / 1000.0; yn = s[cnl].values; z = s[chi].values
    ax = fig.add_subplot(1, 3, k + 1, projection='3d')
    sx = (xb - xb.mean()) / xb.std(); sy = (yn - yn.mean()) / yn.std()
    X = np.column_stack([np.ones_like(sx), sx, sy, sx*sx, sy*sy, sx*sy])
    beta = np.linalg.lstsq(X, z, rcond=None)[0]
    gx = np.linspace(np.percentile(xb,2), np.percentile(xb,98), 24); gy = np.linspace(np.percentile(yn,2), np.percentile(yn,98), 24)
    XX, YY = np.meshgrid(gx, gy)
    tx = (XX - xb.mean()) / xb.std(); tyy = (YY - yn.mean()) / yn.std()
    XG = np.column_stack([np.ones(XX.size), tx.ravel(), tyy.ravel(), (tx**2).ravel(), (tyy**2).ravel(), (tx*tyy).ravel()])
    ZZ = (XG @ beta).reshape(XX.shape)
    ax.plot_surface(XX, YY, ZZ, cmap=cmap, linewidth=0.1, edgecolor='#6A6A6A', shade=False)
    ax.scatter(xb, yn, z, s=0.8, color=RED, alpha=0.35, depthshade=False)
    ax.set_title('%s (N=%d)' % (ty, len(s)), fontsize=4.8); ax.view_init(24, -56); ax.set_box_aspect((1,1,0.6))
fig.tight_layout(); fig.savefig('/content/creads/figs_repro_fig6.png', dpi=200); plt.show()
_pred_map = pd.read_csv(rpd('predicted_584_regions_with_cluster_regimes.csv')).set_index('region_id')['HDI_predicted']
lab['HDI_predicted'] = lab['region_id'].map(_pred_map)
assert lab['HDI_predicted'].notna().all(), 'Missing supplied deployment predictions'
lab['admin'] = lab.groupby('ISO3').HDI.transform('mean')
lab['pred_dev'] = lab.HDI_predicted - lab.groupby('ISO3').HDI_predicted.transform('mean')
lab['hybrid'] = lab.admin + lab.pred_dev
srt = lab.sort_values('HDI', kind='mergesort'); cs = srt.pop_sum.cumsum()/srt.pop_sum.sum()
lab['_poor'] = lab.region_id.isin(set(srt[cs.shift(fill_value=0) < 0.20].region_id))
ppop = lab.loc[lab._poor, 'pop_sum'].sum()
covs = np.arange(0.05, 0.501, 0.025); rng = np.random.default_rng(0)
curves = {nm: np.zeros(len(covs)) for nm in ['admin','pred','hybrid']}
for _ in range(200):
    for nm, col in [('admin','admin'), ('pred','HDI_predicted'), ('hybrid','hybrid')]:
        d = lab.assign(_j=rng.random(len(lab))).sort_values([col,'_j'], kind='mergesort')
        c = d.pop_sum.cumsum()/d.pop_sum.sum(); p = d['_poor'].values
        for kk, cv in enumerate(covs):
            mm = (c.shift(fill_value=0) < cv).values
            curves[nm][kk] += d['pop_sum'].values[mm][p[mm]].sum()/ppop*100
for nm in curves: curves[nm] /= 200
fig, ax = plt.subplots(figsize=(3.4, 2.2))
for nm, cc, ls, lw2, lbl in [('admin',GREY,(0,(5,3)),1.3,'Administrative ranking'), ('pred',BLUE,(0,(4,2.4)),2.4,'Prediction-only ranking'), ('hybrid',RED,'-',1.2,'Administrative + predicted deviation')]:
    kw = dict(markevery=5, marker='o', ms=2.8, mfc='white', mec=BLUE, mew=0.7) if nm == 'pred' else {}
    ax.plot(covs*100, curves[nm], color=cc, ls=ls, lw=lw2, label=lbl, **kw)
for cv in [20, 25, 30]: ax.axvline(cv, color='#BBBBBB', lw=0.5, ls=':')
ax.set_xlabel('Coverage budget (%)', fontsize=6.5); ax.set_ylabel('Recall of poorest 20% (%)', fontsize=6.5)
ax.grid(axis='y', color='#E5E5E5', lw=0.4); ax.set_axisbelow(True)
for cv2 in [20, 25, 30]: ax.text(cv2, 2, str(cv2) + ('%' if cv2 == 20 else ''), fontsize=4.2, ha='center', color='#777777')
ax.legend(fontsize=5, frameon=False)
fig.tight_layout(); fig.savefig('/content/creads/figs_repro_fig8B.png', dpi=200); plt.show()
print('Fig. 4 / 6 / 8B reproduced -> figs_repro_*.png')